# Day 24 — Lecture 3: Other FE Techniques

> Source: Apna College Prime AI/ML • Day 24 Lecture 3 • Video Duration: `08:19`
> Core Objective: Derived features, Interaction features (`age_smoker`, `bmi_smoker`), Feature Scaling aur Feature Selection seekhna.
> Key Parameters: `age_smoker = age * smoker`, `bmi_smoker = bmi * smoker`
> Dataset Used: `insurance.csv`
> Core Concept: Existing features ko combine ya transform karke naye interaction features banana model ka $R^2$ score `0.7835` se seedha `0.8652` tak boost karta hai

## 🛠️ Step 0: Setup & Base Encoded Data

Chalo `insurance.csv` load karke base categorical encoding (`drop_first=True`, `sex`, `smoker`) ready karte hain.

In [ ]:
# 1. Libraries import karo
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.preprocessing import StandardScaler, MinMaxScaler

# 2. Dataset load karke base encoding karo
insurance_data = pd.read_csv("insurance.csv")
X = insurance_data.drop(columns=["charges"])
y = insurance_data["charges"]
X = pd.get_dummies(X, columns=["region"], drop_first=True, dtype=int)
X["sex"] = X["sex"].map({"female": 1, "male": 0})
X["smoker"] = X["smoker"].map({"yes": 1, "no": 0})
print("Base X shape:", X.shape)

## 📍 Step 1: Technique 1 — Derived Features (BMI Categories)

Existing numeric column se naya domain-based column banana **Derived Feature** kehlata hai.

| BMI Range | Category | Numeric Code |
|---|---|---|
| `< 18` | Underweight | `0` |
| `18` to `25` | Normal | `1` |
| `> 25` | Overweight | `2` |

In [ ]:
# 1. Derived feature logic test karne ke liye BMI function banao
def bmi_to_code(val):
    if val < 18:
        return 0
    elif val <= 25:
        return 1
    return 2

# 2. Copy dataframe par derived category count check karo
bmi_codes = insurance_data["bmi"].apply(bmi_to_code)
print("BMI Category Counts (0=Under, 1=Normal, 2=Over):")
print(bmi_codes.value_counts().sort_index())

## 🎯 Step 2: Technique 2 — Interaction Features (`age_smoker`, `bmi_smoker`)

Dekho real-life insurance insight: Agar user ki `age` ya `bmi` zyada ho **AUR** wo `smoker` bhi ho (`1`), toh medical charges bohot badhte hain.
- **Non-Smoker (`smoker = 0`):** Interaction value automatically `0` ho jati hai.
- **Smoker (`smoker = 1`):** Model ko `age` aur `bmi` ka extra combined impact milta hai.

In [ ]:
# 1. Do naye interaction features create karo
X["age_smoker"] = X["age"] * X["smoker"]
X["bmi_smoker"] = X["bmi"] * X["smoker"]

# 2. Non-smoker (0) vs Smoker (1) rows par interaction impact dekho
print(X[["age", "bmi", "smoker", "age_smoker", "bmi_smoker"]].head(5))

## 📈 Step 3: Model Evaluation — $R^2$ Boost (`0.7835` → `0.8652`)

Chalo interaction features ke saath `LinearRegression()` train karke dekhte hain kitna bada jump milta hai.

| Feature Set | Test $R^2$ Score |
|---|---|
| Day 23 Baseline (No region) | `0.78113` |
| One-Hot Encoding (`region`) | `0.78359` |
| + Interaction Features (`age_smoker`, `bmi_smoker`) | `0.86522` |

In [ ]:
# 1. Interaction features ke saath train-test split karo
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# 2. LinearRegression train aur predict karo
model = LinearRegression()
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

# 3. Naya improved R-squared score print karo
r2 = r2_score(y_test, y_pred)
print("r2 score:", r2)

## ⚙️ Step 4: Technique 3 — Scaling Numeric Features

Jab ek column `0/1` range me ho (`smoker`) aur doosra column hazaaron/lakhon me (`salary` ya `age_smoker`), toh bade numbers model ko dominate kar sakte hain.
- **Normalization (`MinMaxScaler`):** Values ko `0` se `1` range me lata hai.
- **Standardization (`StandardScaler`):** Data ko mean `0` aur standard deviation `1` par scale karta hai.

In [ ]:
# 1. Raw feature ranges check karo (min aur max difference)
print("Before Scaling - age_smoker max:", X["age_smoker"].max(), "| smoker max:", X["smoker"].max())

# 2. Standardization aur Normalization demo dekho
std_vals = StandardScaler().fit_transform(X[["age", "bmi", "age_smoker"]])
norm_vals = MinMaxScaler().fit_transform(X[["age", "bmi", "age_smoker"]])

# 3. Scaled ranges verify karo
print("Standardized mean ~", round(std_vals.mean(), 4), "| Normalized min-max:", norm_vals.min(), "to", norm_vals.max())

## 🏆 Step 5: Technique 4 — Feature Selection & Hit-and-Trial

Yaad rakhna: Har feature model me barabar contribute nahi karta — irrelevant columns (jaise student score prediction me student ka naam) remove karne padte hain.
- **ML Reality:** Koi 100% guarantee rule nahi hota; best features chunne ke liye domain insight + hit-and-trial zaroori hai.

In [ ]:
# 1. Sabhi numeric features ka charges ke saath correlation nikalo
full_with_target = X.copy()
full_with_target["charges"] = y
corrs = full_with_target.corr()["charges"].sort_values(ascending=False)

# 2. Top impactful features vs low-impact features dekho
print("Feature Correlations with Medical Charges:")
print(corrs)